In [4]:
"""
Parameter counting script for all VGG19-BN model configurations.
Mirrors the exact freeze/unfreeze logic from src/models/vgg19.py.
Designed for Jupyter notebook display via pandas DataFrames.
"""
import sys
import os
sys.path.insert(0, os.getcwd())

import contextlib
import io
import torch
import torch.nn as nn
import pandas as pd
from IPython.display import display, HTML


# ── Helpers ───────────────────────────────────────────────────────────────────

def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, total - trainable


def component_breakdown(model, model_type):
    if model_type == "baseline":
        m = model.model.vgg
        bb_t  = sum(p.numel() for p in m.features.parameters()   if p.requires_grad)
        cls_t = sum(p.numel() for p in m.classifier.parameters() if p.requires_grad)
        return bb_t, 0, cls_t, 0

    elif model_type == "attention":
        inner = model.model
        bb_t  = sum(p.numel() for p in inner.features.parameters()   if p.requires_grad)
        att_t = sum(p.numel() for p in inner.attention.parameters()  if p.requires_grad)
        cls_t = sum(p.numel() for p in inner.classifier.parameters() if p.requires_grad)
        return bb_t, att_t, cls_t, 0

    elif model_type == "lora":
        bb_t   = sum(p.numel() for n, p in model.features.named_parameters()
                     if p.requires_grad and 'lora_' not in n)
        lora_t = sum(p.numel() for n, p in model.features.named_parameters()
                     if p.requires_grad and 'lora_' in n)
        cls_t  = sum(p.numel() for p in model.classifier.parameters() if p.requires_grad)
        return bb_t, 0, cls_t, lora_t


def fmt(n):
    """Format an integer with comma separators."""
    return f"{n:,}"


def build_model_silently(cls, kwargs):
    """Instantiate a model while suppressing stdout."""
    with contextlib.redirect_stdout(io.StringIO()):
        return cls(pretrained=False, **kwargs)


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    from src.models.vgg19 import (
        VGG19Baseline,
        VGG19SoftmaxAttention,
        VGG19SEAttention,
        VGG19CBAMAttention,
        VGG19SelfAttention,
        VGG19LoRA,
        SoftmaxAttention, SEAttention, CBAM, SelfAttention,
    )

    configs = [
        # (label, model_class, kwargs, model_type)
        ("1. Frozen baseline (no attn)",  VGG19Baseline,         {},                                           "baseline"),
        ("2. Frozen + Softmax",           VGG19SoftmaxAttention, {},                                           "attention"),
        ("3. Frozen + SE",                VGG19SEAttention,      {"reduction": 16},                            "attention"),
        ("4. Frozen + CBAM",              VGG19CBAMAttention,    {},                                           "attention"),
        ("5. Frozen + Self-Attention",    VGG19SelfAttention,    {},                                           "attention"),
        ("6. Finetune=40 (no attn)¹",    VGG19Baseline,         {},                                           "baseline"),
        ("7a. Finetune=40 + Softmax",     VGG19SoftmaxAttention, {"unfreeze_from_layer": 40},                  "attention"),
        ("7b. Finetune=40 + SE",          VGG19SEAttention,      {"reduction": 16, "unfreeze_from_layer": 40}, "attention"),
        ("7c. Finetune=40 + CBAM",        VGG19CBAMAttention,    {"unfreeze_from_layer": 40},                  "attention"),
        ("7d. Finetune=40 + Self-Attn",   VGG19SelfAttention,    {"unfreeze_from_layer": 40},                  "attention"),
        ("8. Finetune=35 + Softmax",      VGG19SoftmaxAttention, {"unfreeze_from_layer": 35},                  "attention"),
        ("9. LoRA rank=8",                VGG19LoRA,             {"rank": 8},                                  "lora"),
    ]

    # ── Build models & collect rows ───────────────────────────────────────────
    rows = []
    for label, cls, kwargs, mtype in configs:
        model = build_model_silently(cls, kwargs)
        total, trainable, frozen = count_params(model)
        bb_t, att_t, cls_t, lora_t = component_breakdown(model, mtype)
        rows.append({
            "Configuration":      label,
            "Total":              total,
            "Trainable":          trainable,
            "Frozen":             frozen,
            "BB Trainable":       bb_t,
            "Attn Trainable":     att_t,
            "Head Trainable":     cls_t,
            "LoRA Trainable":     lora_t,
        })

    # ── Table 1 ───────────────────────────────────────────────────────────────
    df1 = pd.DataFrame(rows).set_index("Configuration")
    int_cols = ["Total", "Trainable", "Frozen",
                "BB Trainable", "Attn Trainable", "Head Trainable", "LoRA Trainable"]

    display(HTML("<h3>Table 1 — Parameter Counts by Configuration</h3>"
                 "<small>¹ VGG19Baseline has no <code>unfreeze_from_layer</code> arg — "
                 "counts are identical to frozen baseline.</small>"))
    display(df1[int_cols].style
            .format({c: "{:,}" for c in int_cols})
            .set_table_styles([
                {"selector": "th", "props": [("text-align", "right"), ("white-space", "nowrap")]},
                {"selector": "td", "props": [("text-align", "right"), ("font-family", "monospace")]},
                {"selector": "th.col_heading", "props": [("min-width", "110px")]},
            ])
            .set_caption("All counts are exact integers. pretrained=False (weights do not affect parameter count).")
    )

    # ── Table 2 ───────────────────────────────────────────────────────────────
    att_rows = []
    for name, mod in [
        ("Softmax",        SoftmaxAttention(512)),
        ("SE (r=16)",      SEAttention(512, reduction_ratio=16)),
        ("CBAM (r=16,k=7)",CBAM(512, reduction_ratio=16, kernel_size=7)),
        ("Self-Attention",  SelfAttention(512)),
    ]:
        att_rows.append({"Attention Type": name,
                         "Parameters": sum(p.numel() for p in mod.parameters())})

    df2 = pd.DataFrame(att_rows).set_index("Attention Type")
    display(HTML("<h3>Table 2 — Attention Module Parameters (isolated, 512 channels)</h3>"))
    display(df2.style
            .format({"Parameters": "{:,}"})
            .set_table_styles([
                {"selector": "td", "props": [("text-align", "right"), ("font-family", "monospace")]},
                {"selector": "th", "props": [("text-align", "left")]},
            ])
    )

    # ── Delta table ───────────────────────────────────────────────────────────
    row_40 = next(r for r in rows if r["Configuration"].startswith("7a."))
    row_35 = next(r for r in rows if r["Configuration"].startswith("8."))
    delta  = row_35["Trainable"] - row_40["Trainable"]
    delta_bb = row_35["BB Trainable"] - row_40["BB Trainable"]

    df3 = pd.DataFrame([
        {"Metric": "Trainable params (unfreeze=40)", "Value": row_40["Trainable"]},
        {"Metric": "Trainable params (unfreeze=35)", "Value": row_35["Trainable"]},
        {"Metric": "Delta (trainable)",              "Value": delta},
        {"Metric": "Delta (backbone only)",          "Value": delta_bb},
        {"Metric": "Delta in millions",              "Value": f"{delta / 1e6:.6f} M"},
    ]).set_index("Metric")

    display(HTML("<h3>Delta: unfreeze=35 vs unfreeze=40 (Softmax Attention)</h3>"))
    display(df3.style
            .set_table_styles([
                {"selector": "td", "props": [("text-align", "right"), ("font-family", "monospace")]},
                {"selector": "th", "props": [("text-align", "left")]},
            ])
    )

    # ── Layer boundary reference ───────────────────────────────────────────────
    from torchvision.models import vgg19_bn
    with contextlib.redirect_stdout(io.StringIO()):
        vgg = vgg19_bn(weights=None)

    layer_rows = []
    for i, layer in enumerate(vgg.features):
        if i >= 33:
            params = sum(p.numel() for p in layer.parameters())
            layer_rows.append({
                "Index": i,
                "Layer Type": layer.__class__.__name__,
                "Params": params,
                "Unfrozen by =35": "✓" if i >= 35 and params > 0 else ("—" if params == 0 else ""),
                "Unfrozen by =40": "✓" if i >= 40 and params > 0 else ("—" if params == 0 else ""),
            })

    df4 = pd.DataFrame(layer_rows).set_index("Index")
    display(HTML("<h3>Backbone Layer Indices (VGG19-BN features[33:53])</h3>"))
    display(df4.style
            .format({"Params": "{:,}"})
            .set_table_styles([
                {"selector": "td", "props": [("text-align", "right"), ("font-family", "monospace")]},
                {"selector": "th", "props": [("text-align", "center")]},
                {"selector": "td:nth-child(2)", "props": [("text-align", "left")]},
            ])
    )


main()


,Total,Trainable,Frozen,BB Trainable,Attn Trainable,Head Trainable,LoRA Trainable
Configuration,,,,,,,
1. Frozen baseline (no attn),"139,597,636","16,388","139,581,248",0,0,"16,388",0
2. Frozen + Softmax,"156,526,380","12,848,132","143,678,248",0,512,"12,847,620",0
3. Frozen + SE,"156,558,636","12,880,388","143,678,248",0,"32,768","12,847,620",0
4. Frozen + CBAM,"156,559,246","12,880,998","143,678,248",0,"33,378","12,847,620",0
5. Frozen + Self-Attention,"156,854,189","13,175,941","143,678,248",0,"328,321","12,847,620",0
6. Finetune=40 (no attn)¹,"139,597,636","16,388","139,581,248",0,0,"16,388",0
7a. Finetune=40 + Softmax,"156,526,380","22,291,460","134,234,920","9,443,328",512,"12,847,620",0
7b. Finetune=40 + SE,"156,558,636","22,323,716","134,234,920","9,443,328","32,768","12,847,620",0
7c. Finetune=40 + CBAM,"156,559,246","22,324,326","134,234,920","9,443,328","33,378","12,847,620",0


,Parameters
Attention Type,
Softmax,512
SE (r=16),"32,768"
"CBAM (r=16,k=7)","33,378"
Self-Attention,"328,321"


,Value
Metric,
Trainable params (unfreeze=40),22291460
Trainable params (unfreeze=35),24652292
Delta (trainable),2360832
Delta (backbone only),2360832
Delta in millions,2.360832 M


,Layer Type,Params,Unfrozen by =35,Unfrozen by =40
Index,,,,
33,Conv2d,"2,359,808",,
34,BatchNorm2d,"1,024",,
35,ReLU,0,—,—
36,Conv2d,"2,359,808",✓,
37,BatchNorm2d,"1,024",✓,
38,ReLU,0,—,—
39,MaxPool2d,0,—,—
40,Conv2d,"2,359,808",✓,✓
41,BatchNorm2d,"1,024",✓,✓
